# FINAL PROJECT: 10-Model Explicit OOF Stacking (PRO VERSION)
## Objective: Optimized Batched Inference & Clean Callbacks

**Project Architecture:**
- **Base Models (10):** NB, LogReg, SVM, RF, MLP, TextCNN, LSTM, BiLSTM, BiGRU, DistilBERT.
- **Optimization:** Batched BERT Inference & Localized EarlyStopping.
- **Stacking Logic:** 5-Fold Stratified Cross-Validation (Out-of-Fold).
- **Meta-Learner:** XGBoost (Explicitly trained on the OOF matrix).

---

## PART 1: GLOBAL SETUP & IMPORTS

In [ ]:
import os, re, string, random, zipfile, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Embedding, LSTM, GRU, Dense, Dropout, SpatialDropout1D, Bidirectional,
    Input, Concatenate, GlobalAveragePooling1D, GlobalMaxPooling1D, Conv1D, Layer
)
from tensorflow.keras.callbacks import EarlyStopping

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import xgboost as xgb

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError:
    nltk.download('vader_lexicon')

SEED = 42
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Global Setup Complete. Device: {DEVICE}")

## PART 2: DATA LOADING & EXPLICIT CLEANING

In [ ]:
def extract_meta_features(df):
    df = df.copy()
    df['exclamation_count'] = df['text'].apply(lambda x: str(x).count('!'))
    df['question_count'] = df['text'].apply(lambda x: str(x).count('?'))
    df['is_all_caps'] = df['text'].apply(lambda x: 1 if str(x).isupper() and len(str(x)) > 5 else 0)
    df['char_cnt'] = df['text'].apply(lambda x: len(str(x)))
    df['word_cnt'] = df['text'].apply(lambda x: len(str(x).split()))
    platforms = r'github|slack|coursera|udemy|paystack|railway|netlify|heroku|mtn|airtel|gmail|whatsapp'
    alerts = r'invoice|billing|terminate|security|alert|reminder|approved|successful|failed|payment'
    df['has_platform_mention'] = df['text'].apply(lambda x: 1 if re.search(platforms, str(x).lower()) else 0)
    df['has_service_alert'] = df['text'].apply(lambda x: 1 if re.search(alerts, str(x).lower()) else 0)
    return df

def surgical_cleaner(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+|@\w+', '', text)
    text = re.sub(r'<.*?>', '', text)
    prefixes = r'^(mtn|airtel|github|slack|gmail|whatsapp|udemy|service termination notice|billing alert|security alert|reminder|alert|forwarded message|from:|to:|subject:|date:|sent:):\s*'
    text = re.sub(prefixes, '', text, flags=re.IGNORECASE)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text if text else "notification"

print("Loading datasets...")
train_df = pd.read_csv('../data/processed/processed_training_dataset.csv').dropna()
test_df  = pd.read_csv('../data/processed/student_test_dataset.csv').dropna()

train_df = extract_meta_features(train_df)
test_df  = extract_meta_features(test_df)

train_df['clean'] = train_df['text'].apply(surgical_cleaner)
test_df['clean']  = test_df['text'].apply(surgical_cleaner)

label_map = {"Negative": 0, "Neutral": 1, "Positive": 2}
train_df['label'] = train_df['sentiment'].map(label_map)
test_df['label']  = test_df['sentiment'].map(label_map)

y_train = train_df['label'].values
y_test  = test_df['label'].values

print(f"Cleaning complete. Training Samples: {len(train_df)}")

## PART 3: EXPLICIT FEATURE PREPARATION

In [ ]:
print("Scaling Metadata Features...")
meta_cols = ['exclamation_count', 'question_count', 'is_all_caps', 'char_cnt', 'word_cnt', 'has_platform_mention', 'has_service_alert']
scaler = StandardScaler()
X_train_meta = scaler.fit_transform(train_df[meta_cols])
X_test_meta  = scaler.transform(test_df[meta_cols])

print("Preparing GloVe Embedding Matrix...")
VOCAB_SIZE, MAX_LEN = 20000, 150
dl_tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
dl_tokenizer.fit_on_texts(train_df['clean'])

glove_path = 'glove.6B.100d.txt'
if not os.path.exists(glove_path):
    urllib.request.urlretrieve('https://nlp.stanford.edu/data/glove.6B.zip', 'glove.6B.zip')
    with zipfile.ZipFile('glove.6B.zip', 'r') as z: z.extract(glove_path)

embeddings_index = {}
with open(glove_path, encoding='utf8') as f:
    for line in f:
        v = line.split()
        embeddings_index[v[0]] = np.asarray(v[1:], dtype='float32')

embedding_matrix = np.zeros((VOCAB_SIZE, 100))
for word, i in dl_tokenizer.word_index.items():
    if i < VOCAB_SIZE:
        vec = embeddings_index.get(word)
        if vec is not None: embedding_matrix[i] = vec

## PART 4: MODEL HELPER FUNCTIONS

In [ ]:
def get_deep_model(m_type='cnn'):
    inp = Input(shape=(MAX_LEN,))
    x = Embedding(VOCAB_SIZE, 100, weights=[embedding_matrix])(inp)
    if m_type == 'cnn':
        x = Conv1D(128, 5, activation='relu')(x)
        x = GlobalMaxPooling1D()(x)
    elif m_type == 'lstm':
        x = LSTM(128)(x)
    elif m_type == 'bilstm':
        x = Bidirectional(LSTM(64))(x)
    elif m_type == 'bigru':
        x = Bidirectional(GRU(64))(x)
    out = Dense(3, activation='softmax')(x)
    m = Model(inputs=inp, outputs=out)
    m.compile(loss='sparse_categorical_crossentropy', optimizer='adam')
    return m

def train_distilbert(train_txt, train_lbl):
    from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
    tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
    train_enc = tokenizer(train_txt.tolist(), truncation=True, padding=True, max_length=128)
    
    class SentiDS(torch.utils.data.Dataset):
        def __init__(self, enc, lbl):
            self.enc = enc
            self.lbl = lbl
        def __getitem__(self, idx):
            item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
            item['labels'] = torch.tensor(self.lbl[idx])
            return item
        def __len__(self): return len(self.lbl)

    model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=3).to(DEVICE)
    args = TrainingArguments(output_dir='results', epochs=1, per_device_train_batch_size=16, disable_tqdm=True)
    trainer = Trainer(model=model, args=args, train_dataset=SentiDS(train_enc, train_lbl))
    trainer.train()
    return model, tokenizer

## PART 5: THE 10-MODEL EXPLICIT OOF LOOP

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_train = np.zeros((len(train_df), 30))
oof_test  = np.zeros((len(test_df), 30))
cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
cw_dict = {i: cw[i] for i in range(3)}

print("Starting Explicit Stacking Loop (No Pipelines)... ")
for fold, (t_idx, v_idx) in enumerate(skf.split(train_df['clean'], y_train)):
    print(f"--- Fold {fold+1} ---")
    # Localized EarlyStopping to reset each fold
    es = EarlyStopping(patience=2, restore_best_weights=True)
    
    # 1. Classical Vectorization
    vec = TfidfVectorizer(max_features=5000)
    vec.fit(train_df['clean'].iloc[t_idx])
    X_t_tfidf = vec.transform(train_df['clean'].iloc[t_idx])
    X_v_tfidf = vec.transform(train_df['clean'].iloc[v_idx])
    X_test_tfidf_f = vec.transform(test_df['clean'])
    
    # 2. NB (0:3)
    nb = MultinomialNB().fit(X_t_tfidf, y_train[t_idx])
    oof_train[v_idx, 0:3] = nb.predict_proba(X_v_tfidf)
    oof_test[:, 0:3] += nb.predict_proba(X_test_tfidf_f) / 5
    
    # 3. LogReg (3:6)
    lr = LogisticRegression(max_iter=1000).fit(X_t_tfidf, y_train[t_idx])
    oof_train[v_idx, 3:6] = lr.predict_proba(X_v_tfidf)
    oof_test[:, 3:6] += lr.predict_proba(X_test_tfidf_f) / 5
    
    # 4. SVM (6:9)
    svm = SVC(C=1.0, kernel='rbf', probability=True, class_weight='balanced').fit(X_t_tfidf, y_train[t_idx])
    oof_train[v_idx, 6:9] = svm.predict_proba(X_v_tfidf)
    oof_test[:, 6:9] += svm.predict_proba(X_test_tfidf_f) / 5
    
    # 5. RF (9:12)
    rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', n_jobs=-1).fit(X_t_tfidf, y_train[t_idx])
    oof_train[v_idx, 9:12] = rf.predict_proba(X_v_tfidf)
    oof_test[:, 9:12] += rf.predict_proba(X_test_tfidf_f) / 5
    
    # 6. MLP (12:15)
    mlp = MLPClassifier(hidden_layer_sizes=(128,64), max_iter=300).fit(X_t_tfidf, y_train[t_idx])
    oof_train[v_idx, 12:15] = mlp.predict_proba(X_v_tfidf)
    oof_test[:, 12:15] += mlp.predict_proba(X_test_tfidf_f) / 5
    
    # Deep Learning Prep
    X_t_seq = pad_sequences(dl_tokenizer.texts_to_sequences(train_df['clean'].iloc[t_idx]), maxlen=MAX_LEN)
    X_v_seq = pad_sequences(dl_tokenizer.texts_to_sequences(train_df['clean'].iloc[v_idx]), maxlen=MAX_LEN)
    X_test_seq_f = pad_sequences(dl_tokenizer.texts_to_sequences(test_df['clean']), maxlen=MAX_LEN)
    
    # 7. CNN (15:18)
    cnn = get_deep_model('cnn')
    cnn.fit(X_t_seq, y_train[t_idx], epochs=10, batch_size=64, verbose=0, callbacks=[es], class_weight=cw_dict)
    oof_train[v_idx, 15:18] = cnn.predict(X_v_seq)
    oof_test[:, 15:18] += cnn.predict(X_test_seq_f) / 5
    
    # 8. LSTM (18:21)
    lstm = get_deep_model('lstm')
    lstm.fit(X_t_seq, y_train[t_idx], epochs=10, batch_size=64, verbose=0, callbacks=[es], class_weight=cw_dict)
    oof_train[v_idx, 18:21] = lstm.predict(X_v_seq)
    oof_test[:, 18:21] += lstm.predict(X_test_seq_f) / 5
    
    # 9. BiLSTM (21:24)
    bilstm = get_deep_model('bilstm')
    bilstm.fit(X_t_seq, y_train[t_idx], epochs=10, batch_size=64, verbose=0, callbacks=[es], class_weight=cw_dict)
    oof_train[v_idx, 21:24] = bilstm.predict(X_v_seq)
    oof_test[:, 21:24] += bilstm.predict(X_test_seq_f) / 5
    
    # 10. BiGRU (24:27)
    bigru = get_deep_model('bigru')
    bigru.fit(X_t_seq, y_train[t_idx], epochs=10, batch_size=64, verbose=0, callbacks=[es], class_weight=cw_dict)
    oof_train[v_idx, 24:27] = bigru.predict(X_v_seq)
    oof_test[:, 24:27] += bigru.predict(X_test_seq_f) / 5
    
    # 11. DistilBERT (27:30)
    bert_m, bert_t = train_distilbert(train_df['clean'].iloc[t_idx], y_train[t_idx])
    bert_m.eval()
    with torch.no_grad():
        # Batched Inference Val (Optimized for Kaggle)
        v_txts = train_df['clean'].iloc[v_idx].tolist()
        v_enc = bert_t(v_txts, return_tensors='pt', padding=True, truncation=True, max_length=128).to(DEVICE)
        v_logits = bert_m(**v_enc).logits
        oof_train[v_idx, 27:30] = torch.softmax(v_logits, dim=-1).cpu().numpy()
        
        # Batched Inference Test (Optimized for Kaggle)
        t_txts = test_df['clean'].tolist()
        t_enc = bert_t(t_txts, return_tensors='pt', padding=True, truncation=True, max_length=128).to(DEVICE)
        t_logits = bert_m(**t_enc).logits
        t_probs = torch.softmax(t_logits, dim=-1).cpu().numpy()
        oof_test[:, 27:30] += (t_probs / 5)
    print(f"Fold {fold+1} Finished.")

## PART 6: THE STACKING MATRIX & META-LEARNER

In [ ]:
print("Stacking OOF Probabilities with Metadata Features...")
X_stack_train = np.hstack([oof_train, X_train_meta])
X_stack_test  = np.hstack([oof_test,  X_test_meta])

print("Training XGBoost Meta-Learner...")
meta_learner = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, objective='multi:softprob')
meta_learner.fit(X_stack_train, y_train)

y_final = meta_learner.predict(X_stack_test)
print("Final Model Execution Complete.")

## PART 7: FINAL EVALUATION

In [ ]:
print("\n" + "="*60)
print("  FINAL 10-MODEL EXPLICIT STACKING - PERFORMANCE")
print("="*60)
print(classification_report(y_test, y_final, target_names=label_map.keys()))

cm = confusion_matrix(y_test, y_final)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='viridis', xticklabels=label_map.keys(), yticklabels=label_map.keys())
plt.title('Final Stacking Confusion Matrix: Student Communication Data')
plt.show()